# Pandas — Example 02: The Working Toolkit

> 📘 **Instructor Curriculum** — M03, Python for Data Science (NumPy and Pandas)

**What this notebook is:** the second Pandas pass. [`example_01.ipynb`](./example_01.ipynb)
covered the shapes — Series, DataFrame, reading a file, selecting, filtering, missing
values, `concat`, and `merge`. This notebook is the working toolkit you reach for on
every real dataset: **inspect → select → clean → transform → aggregate**.

**How to read it:** each Markdown section explains every code cell that follows it, up to
the next section. Two cells here end in errors. They are kept on purpose — both are
errors you will hit again, and the explanation is more useful than a clean cell.

Versions used here: **pandas 3.0.3**, **numpy 2.4.6**, Python 3.14 in `pizza_env`.

---

## Where this fits

NumPy gave you one typed block of numbers. Pandas puts **labels** on it.

```text
   NumPy array                    Pandas DataFrame
   -----------                    ----------------
   [[  'A', 20],                    Name  Age      <- column labels
    [  'B', 25]]                0      A   20      <- row labels (the index)
                                1      B   25

   position only                  labels + position, and one dtype PER COLUMN
```

Two consequences run through this whole notebook:

1. **A DataFrame is a dictionary of columns**, not a grid of cells. Each column is a
   Series with its own dtype. That is why `df['Age']` is the natural move and why one
   column can be `int64` while its neighbour is text.
2. **The index is not row numbers.** It is a set of labels that happen to start as
   `0, 1, 2`. Section 8 changes them, and everything that assumed "row numbers" breaks.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> A DataFrame is close to a `ResultSet` you can keep, re-query, and mutate in memory. The
> mapping is almost one to one: `df[df.Age > 25]` is `WHERE`, `groupby` is `GROUP BY`,
> `merge` is `JOIN`, and the index is a primary key. Most of this notebook is SQL you
> already know, written in Python.

### A note on pandas 3.0 output

Your `info()` and `dtypes` output shows dtype **`str`**. Nearly every tutorial online
shows **`object`** for text columns. Both are correct — pandas 3.0 introduced a real
string dtype. Nothing about your code is wrong; the library moved.


---

## 1. Import

`pd` is the universal alias, exactly like `np`. Never rename it.

Pandas is built on NumPy — every numeric column is a NumPy array underneath, which is why
everything you learned in the NumPy notebooks still applies here.


In [1]:
import pandas as pd

---

## 2. Series — one column with a label for every value

A Series is **one column**: an array of values plus an index of labels.

```text
   index   values
     0       10
     1       20
     2       30       dtype: int64      one type for the whole Series
```

Printing it shows both. The `dtype: int64` line at the bottom is part of the Series, not
decoration — a Series always knows its one type.

| Cell line | Result | What happened |
|---|---|---|
| `print(s)` | the whole Series | index and values, plus the dtype |
| `print(s[0])` | `10` | looked up the **label** `0` |
| `print(s[1:])` | rows 1 and 2 | sliced by **position** |
| `print("Mean", s.mean())` | `20.0` | a whole-column statistic |

### The trap hiding in this cell

`s[0]` and `s[1:]` look like the same kind of access. They are not.

```text
   s[0]     ->  LABEL lookup      "give me the value labelled 0"
   s[1:]    ->  POSITION slice    "give me everything from row 1 onward"
```

You cannot see the difference here, because the default index *is* `0, 1, 2` — label and
position happen to be the same number. Change the index and it splits apart:

```python
s2 = pd.Series([10,20,30], index=[10,11,12])
s2[0]     # KeyError: 0        <- verified. There is no label 0.
s2.iloc[0]  # 10               <- position 0
```

**Say what you mean:** `s.iloc[0]` for position, `s.loc[0]` for label. Section 7 covers
this properly, and section 8 shows it breaking on a real DataFrame.

**Takeaway:** a Series is values plus labels. Plain `s[...]` guesses which one you meant;
`.loc` and `.iloc` do not have to guess.


In [4]:
s=pd.Series([10,20,30])
print(s)
print("---------------")
print(s[0])
print("---------------")
print(s[1:])
print("---------------")
print("Mean",s.mean())

0    10
1    20
2    30
dtype: int64
---------------
10
---------------
1    20
2    30
dtype: int64
---------------
Mean 20.0


---

## 3. DataFrame — and your question: why `df.columns` but `df.head()`?

You asked this in the cell's comments, so this section answers it properly.

```python
df = pd.DataFrame({'Name':['A','B'], 'Age': [20,25]})
```

A dictionary becomes a DataFrame: **each key is a column name, each list is that
column's values**. The index `0, 1` was created for you.

### The answer: attribute vs method

| | Attribute | Method |
|---|---|---|
| Written | `df.columns` — **no** parentheses | `df.head()` — **with** parentheses |
| It is | a fact the object already knows | an action you ask it to perform |
| Can take options? | no | yes — `df.head(3)`, `df.sort_values('Age')` |
| Think of it as | a property of the data | a function that does work |

The rule that decides it:

```text
   Does the answer depend on anything you choose?
        no   ->  attribute      shape, columns, dtypes, index
        yes  ->  method         head(n), describe(), sort_values(by), fillna(value)
```

`df.shape` cannot mean anything except `(2, 2)`. `df.head()` has to be told how many rows
— it just defaults to 5. That is the whole distinction.

### What it looks like when you get it wrong

```python
df.head          # <bound method NDFrame.head of   Name  Age ...>    verified
df.shape()       # TypeError: 'tuple' object is not callable
```

Forgetting the parentheses does not raise an error — it prints the **method object**. If
your output ever starts with `<bound method`, you forgot `()`.

### "How many of this type exist?" — the ones you will actually use

**Attributes (no parentheses):**

| Attribute | Gives you |
|---|---|
| `df.shape` | `(rows, columns)` — a plain tuple |
| `df.columns` | the column names — an `Index` object |
| `df.index` | the row labels |
| `df.dtypes` | the type of each column |
| `df.size` | total number of cells |
| `df.ndim` | 1 for a Series, 2 for a DataFrame |
| `df.empty` | `True` if there are no rows |
| `df.T` | the frame, transposed |
| `df.values` | the underlying NumPy array |

**Methods (parentheses required):**

`head()`, `tail()`, `info()`, `describe()`, `sample()`, `sum()`, `mean()`, `count()`,
`sort_values()`, `groupby()`, `merge()`, `dropna()`, `fillna()`, `astype()`, `rename()`,
`query()`, `apply()`, `to_csv()` — and nearly everything else in this notebook.

Methods vastly outnumber attributes. A short list of attributes describes the frame; the
long list of methods changes it.

**Check any name yourself:**

```python
callable(df.head)        # True  -> it is a method, use ()
callable(df.shape)       # False -> it is an attribute
```

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> `df.shape` is not a stored field. It is a Python **property** — a getter that runs on
> access with the parentheses hidden. In Java you would write `df.getShape()`; Python lets
> the class present the same thing as `df.shape`. So "attribute" here means *looks like a
> field, may be computed*, and that is why it never takes arguments.

Two details in this cell's output worth noticing:

- `df.columns` prints `Index(['Name','Age'], dtype='str')` — column names are themselves
  an indexed pandas object, not a plain list. `list(df.columns)` when you want a list.
- `df.head()` returned both rows. `head` shows the **first 5** and there are only 2, so
  nothing was cut. On a real file, `head()` is how you look without printing a million
  rows.

**Takeaway:** no parentheses means "tell me a fact about this frame". Parentheses mean
"do something". If you see `<bound method`, add the `()`.


In [ ]:
df = pd.DataFrame({'Name':['A','B'], 'Age': [20,25]})
print(df)
print("---------------")
print(df.columns) # Why this hape is anot function 
print("---------------")
print(df.shape)
print("---------------")
print(df.head()) # where as head is a function 
print("---------------")
# explain me what is the difference between them and how many this type odf data exist 

  Name  Age
0    A   20
1    B   25
---------------
Index(['Name', 'Age'], dtype='str')
---------------
(2, 2)
---------------
  Name  Age
0    A   20
1    B   25
---------------


---

## 4. Reading files — and the error in this cell

```python
df1 = pd.read_csv('data.csv')
# FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'
```

This error is correct and it is worth keeping. **The read cell runs before the write cell
that creates the file.** At the moment you ran it, `data.csv` did not exist yet.

### Two things to take from it

**1. A relative path is relative to the working directory,** not to the notebook file.
Usually they are the same folder, but not always — a kernel started elsewhere will look
elsewhere. Settle it in one line when a path surprises you:

```python
from pathlib import Path
Path.cwd()                    # where pandas is actually looking
list(Path.cwd().glob('*.csv'))  # what is actually there
```

**2. The cell stops at the first error.** `read_excel` and `read_json` on the next two
lines never ran. Everything after a raised exception in the same cell is skipped — worth
remembering when a cell has side effects, which is exactly what happens in section 5.

### The readers

| Call | Reads | Needs |
|---|---|---|
| `pd.read_csv` | text, comma-separated | nothing extra |
| `pd.read_excel` | `.xlsx` | the **`openpyxl`** package |
| `pd.read_json` | JSON | nothing extra |

They all return a DataFrame, so whatever the source, the rest of your code is the same.
That is the real point of this cell: **pandas is a single interface over many formats.**

To make this cell run, either move it below the write cell, or create the file first.
`pd.read_csv('data.csv')` works now, because section 5 created it.

**Takeaway:** `read_*` turns a file into a DataFrame. When it cannot find the file, check
the working directory before you doubt the code.


In [7]:
df1 = pd.read_csv('data.csv')
df2 = pd.read_excel('data.xlsx')
df3 = pd.read_json('data.json')

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

---

## 5. Writing files — the cell half-succeeded

```python
df.to_csv('data.csv')      # worked
df.to_excel('data.xlsx')   # ModuleNotFoundError: No module named 'openpyxl'
df.to_json('data.json')    # never ran
```

This is the side-effect case from section 4, live. The cell raised on line 2, so line 3
was skipped — but **line 1 had already written a file to disk**. A failed cell does not
undo what it already did. `data.csv` exists in this folder because of that cell.

Excel is the only format here that needs an extra package:

```text
pip install openpyxl        then restart the kernel
```

### The `index=False` trap — check your `data.csv`

Look at what was actually written:

```text
   ,Name,Age            <- the first column has no name
   0,A,20
   1,B,25
```

`to_csv` writes the index as an unnamed first column by default. Read that file back and
you get a junk column:

```python
pd.read_csv('data.csv').columns
# ['Unnamed: 0', 'Name', 'Age']       verified — your file does this today
```

Do it a few times and you collect `Unnamed: 0`, `Unnamed: 0.1`, and so on. The fix is one
argument:

```python
df.to_csv('data.csv', index=False)     # unless the index is real data
```

Keep the index only when it means something — a date, an ID. A `RangeIndex` of `0,1,2`
means nothing and should not be saved.

**Takeaway:** `to_*` mirrors `read_*`. Write CSV with `index=False`, and remember that a
cell that failed halfway may still have changed things on disk.


In [8]:
df.to_csv('data.csv')
df.to_excel('data.xlsx')
df.to_json('data.json')

ModuleNotFoundError: No module named 'openpyxl'

---

## 6. First look at a frame — `info`, `shape`, `describe`

These three are what you run on any dataset you have not seen before, in this order.

| Call | Answers |
|---|---|
| `df.info()` | What columns exist, what type are they, **how many values are missing**? |
| `df.shape` | How big is it? |
| `df.describe()` | What do the numbers look like? |

### Reading `info()`

```text
   RangeIndex: 2 entries, 0 to 1          how many rows, and the index type
   #   Column  Non-Null Count  Dtype
   0   Name    2 non-null      str        <- pandas 3.0 string dtype
   1   Age     2 non-null      int64
   memory usage: 166.0 bytes
```

**`Non-Null Count` is the column you actually read.** Compare it with the row count: if a
column says `1850 non-null` and there are 2000 rows, you have 150 missing values and
section 11 is your next stop. This one line is the fastest data-quality check in pandas.

### Reading `describe()`

It returned **only `Age`**. Verified: `describe()` skips text columns by default, because
a mean of `'A'` and `'B'` means nothing. Use `df.describe(include='all')` when you want
counts and top values for text too.

The eight rows it gives you:

| Row | Meaning |
|---|---|
| `count` | non-missing values — compare against `shape[0]` |
| `mean`, `std` | average and spread |
| `min`, `max` | the extremes — the fastest way to spot an impossible value |
| `25%`, `50%`, `75%` | quartiles; `50%` is the median |

`std` here is `3.535534`, and this is the NumPy notebook's `ddof` trap in the flesh:
**pandas uses `ddof=1` (sample), NumPy uses `ddof=0` (population).** The same two numbers
give a different answer in each library. Neither is broken; they answer different
questions.

**Takeaway:** `info()` for structure and missing values, `describe()` for the numbers.
Run both before you write any analysis.


In [11]:
df=pd.DataFrame({'Name':['A','B'],'Age':[20,25]})
df.info()
print("---------------")
print(df.shape)
print("---------------")
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Name    2 non-null      str  
 1   Age     2 non-null      int64
dtypes: int64(1), str(1)
memory usage: 166.0 bytes
---------------
(2, 2)
---------------
             Age
count   2.000000
mean   22.500000
std     3.535534
min    20.000000
25%    21.250000
50%    22.500000
75%    23.750000
max    25.000000


---

## 7. `loc` vs `iloc` — label or position

This is the selection pair you will use for the rest of your Pandas life.

```text
   df.loc[...]   ->  by LABEL      the index value, the column name
   df.iloc[...]  ->  by POSITION   0, 1, 2 ... counted from the top
```

Memory hook: **i** in `iloc` is for **integer position**.

| Cell line | Result | Read it as |
|---|---|---|
| `df.loc[0]` | the row labelled `0`, as a Series | one row, turned on its side |
| `df.loc[:, ['Name']]` | a **DataFrame** of one column | all rows, the `Name` column |
| `df.iloc[1]` | the row in **position 1** | the second row |
| `df.iloc[:2]` | rows 0 and 1 | the first two rows |

### Three details in the output

**1. A single row comes back as a Series** — printed vertically, with the column names
becoming its index. `dtype: object` because that row mixes text and a number, and a
Series holds one type. Rows mix types; columns do not. This is why pandas is
column-first.

**2. Brackets change the shape.** Verified:

```python
df.loc[:, 'Name']       # Series     one column, no frame around it
df.loc[:, ['Name']]     # DataFrame  a list of columns -> still a frame
```

A list means "these columns", even when the list has one item. Use `[['A','B']]` when you
want a frame back, plain `['A']` when you want the Series.

**3. The comma is `[rows, columns]`.** `df.loc[:, ['Name']]` reads "every row, the Name
column". The `:` is the same slice syntax as plain Python.

### The one difference that will catch you

```text
   df.iloc[:2]   ->  positions 0, 1        stop EXCLUDED, like Python
   df.loc[:2]    ->  labels   0, 1, 2      stop INCLUDED
```

`loc` includes the end label. It has to — with labels like `'Jan'` to `'Mar'`, "up to but
not including Mar" would be unusable. It is a deliberate difference, not an inconsistency.

**Takeaway:** `loc` is labels and includes the end; `iloc` is positions and excludes it.
Prefer them over plain `df[...]` — they say which one you meant.


In [13]:
df=pd.DataFrame({'Name':['A','B','C'],'Age':[20,25,30]})
print(df.loc[0])
print("---------------")
print(df.loc[:,['Name']])
print("---------------")
print(df.iloc[1])
print("---------------")
print(df.iloc[:2])
print("---------------")

Name     A
Age     20
Name: 0, dtype: object
---------------
  Name
0    A
1    B
2    C
---------------
Name     B
Age     25
Name: 1, dtype: object
---------------
  Name  Age
0    A   20
1    B   25
---------------


---

## 8. Changing the index — the experiment that proves the point

These four cells are one experiment, and it is the most useful thing in the notebook.
Section 7's labels and positions matched, so nothing could go wrong. Here they are pulled
apart:

```python
df.index = [101, 102, 103]
```

```text
   position:   0     1     2         <- iloc uses these, always 0,1,2...
   label:    101   102   103         <- loc uses these, they are now yours
```

| Cell | Call | Result | Why |
|---|---|---|---|
| A | `df.index = [101,102,103]` | rows now labelled 101-103 | you replaced the labels; the data never moved |
| B | `df.loc[102]` | row B ✅ | `102` **is** a label |
| C | `df.loc[1]` | **`KeyError: 1`** ❌ | there is no label `1` any more |
| D | `df.iloc[1]` | row B ✅ | position 1 still exists — positions cannot be renamed |

Cell C is the valuable one. `KeyError: 1` does not mean "row 1 is missing" — it means
**"there is no label 1"**. Once you read `KeyError` as "that label does not exist", this
whole family of errors becomes obvious.

Notice cell D's output: it prints `Name: 102`. `iloc` found it by position and then
reported its **label**. Position gets you there; the label is what it is called.

### Where this bites in real work

Every one of these leaves you with an index that is no longer `0,1,2`:

```python
df_filtered = df[df.Age > 25]      # keeps the ORIGINAL labels — gaps appear
df_sorted   = df.sort_values('Age')  # labels travel with their rows
df.set_index('customer_id')        # you chose a meaningful key
```

Filter a frame, then ask for `.loc[0]`, and you get a `KeyError` if row 0 was filtered
out. That is the single most common Pandas confusion, and this cell is it in miniature.

The fix when you genuinely want fresh numbering:

```python
df_filtered = df[df.Age > 25].reset_index(drop=True)
```

`drop=True` throws the old labels away instead of keeping them as a new column.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> Treat the index as a **primary key**, not a row number. `df.set_index('order_id')` makes
> lookups by ID direct, and `merge` can join on it. The moment you think of the index as
> "which row is this", you are one filter away from a `KeyError`.

**Takeaway:** positions are always `0,1,2...`; labels are whatever the index says. After
any filter or sort, the two have parted ways.


In [14]:
df.index = [101,102,103]
print(df)

    Name  Age
101    A   20
102    B   25
103    C   30


In [15]:
df.loc[102]

Name     B
Age     25
Name: 102, dtype: object

In [16]:
df.loc[1]

KeyError: 1

In [17]:
df.iloc[1]

Name     B
Age     25
Name: 102, dtype: object

---

## 9. Filtering with a boolean mask

The NumPy mask from `example_02.ipynb`, now with column names.

```python
df[df['Age'] > 25]
```

It runs in two steps, and it helps to see them separately:

```text
   step 1   df['Age'] > 25   ->   0  False        a Series of True/False,
                                  1  False        one per row
                                  2   True
                                  3   True

   step 2   df[ that mask ]  ->   keep only the rows where it says True
```

| Cell line | Result | Note |
|---|---|---|
| `df[df['Age']>25]` | rows 2 and 3 | **the index kept its original labels — 2 and 3, not 0 and 1** |
| `df[df['Age']==30]` | row 2 | `==` for equality, not `=` |

That first note is section 8's lesson arriving on schedule. Filtering does not renumber
anything. `df[df['Age']>25].loc[0]` is a `KeyError` waiting to happen.

### Combining conditions

Same rule as NumPy — `&`, `|`, `~`, and keep the brackets:

```python
df[(df['Age'] > 25) & (df['Age'] < 35)]     # correct
df[df['Age'] > 25 and df['Age'] < 35]       # ValueError: truth value is ambiguous
```

`and` wants one true-or-false answer; a column of four booleans cannot give one.

**Takeaway:** a condition on a column gives a mask; the mask picks rows. The surviving
rows keep their original labels.


In [18]:
df=pd.DataFrame({'Age':[20,25,30,35]})
print(df[df['Age']>25])
print(df[df['Age']==30])

   Age
2   30
3   35
   Age
2   30


---

## 10. `query` — the same filter, written as a sentence

```python
df.query('Age>25')          # identical result to df[df['Age']>25]
```

Both cells print the same rows. `query` takes the condition as a **string** and lets you
name columns bare — no `df[...]` repeated on every term.

| | Mask | `query` |
|---|---|---|
| Reads | `df[(df.Age>25) & (df.City=='Pune')]` | `df.query("Age>25 and City=='Pune'")` |
| `and` / `or` | not allowed — use `&`, `\|` | **allowed**, plain English |
| Column names with spaces | fine | need backticks: `` `Total Sales` `` |
| Checked when? | at parse time, by Python | at run time, inside the string |
| Use a Python variable | directly | prefix with `@`: `query('Age > @cutoff')` |

The trade: `query` reads better as conditions pile up, but a typo inside the string is
only found when the line runs, and your editor cannot help you. Masks are safer for code
that has to hold up; `query` is pleasant for exploring.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> This is the same trade-off as a string SQL query versus a typed criteria builder.
> Readable versus checkable. Same call, same reasons.

**Takeaway:** `query` and the mask do the same job. Use whichever makes the line easier to
read, and remember `and`/`or` only work inside `query`.


In [19]:
df=pd.DataFrame({'Age':[20,25,30,35]})
print(df.query('Age>25'))
print(df.query('Age==30'))

   Age
2   30
3   35
   Age
2   30


---

## 11. Missing values

Real data has holes. Pandas marks them `NaN` and gives you four tools.

| Call | Returns | Use it to |
|---|---|---|
| `df.isnull()` | a True/False frame — **True where missing** | find and count holes |
| `df.notnull()` | the exact opposite | keep what is present |
| `df.fillna(0)` | a copy with holes filled | replace |
| `df.dropna()` | a copy with those rows removed | delete |

### The detail everyone misses in this output

```text
   input    {'A':[1, None, 3]}          integers, with one gap
   output   0    1.0
            2    3.0                    <- 1.0 and 3.0, not 1 and 3
```

**The column became `float64`.** `NaN` is a floating-point value, and NumPy's `int64` has
no way to represent "missing". So the moment one value goes missing, the whole column is
promoted to float. If a column of counts or IDs suddenly prints with `.0`, you have
missing data upstream.

Also note `dropna()` returned labels `0` and `2` — the gap at 1 stays gone. Section 8
again.

### Counting them — the line you will actually use

```python
df.isnull().sum()        # missing values per column
```

`isnull()` gives True/False, `True` counts as 1, so the sum is the count per column. Run
this on every new dataset, right after `info()`.

### `fillna(0)` is a decision, not a default

Filling with `0` says "the value was zero". Often it was not — it was unknown. For a
salary column, `0` drags the mean down and quietly corrupts every statistic after it.
Common alternatives:

```python
df.fillna(df.mean())        # numeric, roughly symmetric
df.fillna(df.median())      # numeric with outliers — safer
df.ffill()                  # time series: carry the last known value forward
```

Choosing the fill **is** the analysis. It is called imputation, and it comes back in the
ML phase.

### None of these change `df`

`fillna` and `dropna` return **copies**. `df` still holds the `NaN` after both cells.
Reassign when you mean it:

```python
df = df.fillna(0)
```

**Takeaway:** find holes with `isnull().sum()`, then decide — fill or drop. The decision
matters more than the syntax, and one missing value turns an int column into floats.


In [20]:
df=pd.DataFrame({'A':[1,None,3]})
print(df.isnull())
print(df.notnull())
print(df.fillna(0))
print(df.dropna())

       A
0  False
1   True
2  False
       A
0   True
1  False
2   True
     A
0  1.0
1  0.0
2  3.0
     A
0  1.0
2  3.0


---

## 12. Duplicates

```text
   A = [1, 1, 2, 2]

   df.duplicated()        ->  [False, True, False, True]
   df.drop_duplicates()   ->  rows 0 and 2
```

`duplicated()` marks a row `True` when **an identical row appeared earlier**. The first
time a value shows up it is `False` — it is not a duplicate of anything yet. That is why
the answer alternates here rather than marking all four.

`drop_duplicates()` keeps the first of each group and returns a copy.

### The two arguments worth knowing now

```python
df.drop_duplicates(subset=['customer_id'])   # duplicate = same ID, other columns ignored
df.drop_duplicates(keep='last')              # keep the newest row instead of the oldest
```

By default a row must match on **every column** to count as a duplicate. In real data
that is rarely what you mean — one different timestamp makes two rows for the same
customer look unique. `subset` is how you say what "the same" means.

`keep='last'` is the one you want when later rows are more recent.

**Takeaway:** `duplicated()` flags repeats after the first. Decide what "duplicate" means
with `subset` before you drop anything.


In [21]:
df=pd.DataFrame({'A':[1,1,2,2]})
print(df.duplicated())
print(df.drop_duplicates())

0    False
1     True
2    False
3     True
dtype: bool
   A
0  1
2  2


---

## 13. `rename` — changing a column name

```python
df.rename(columns={'Age':'Student_Age'})
```

The dictionary reads `{old: new}`. Only the keys you list change; everything else is left
alone, which is why `rename` is safer than assigning to `df.columns` — that requires the
full list in the right order.

**This returns a copy.** `df` still has `Age` after the cell. Keep the result:

```python
df = df.rename(columns={'Age':'Student_Age'})
```

`inplace=True` also exists and is covered in `example_01`. Prefer reassignment: it is
explicit, it chains, and `inplace` is on its way out of pandas.

Renaming rows works the same way with `index=` instead of `columns=`.

**Takeaway:** `rename(columns={old: new})` touches only the names you list, and hands back
a copy.


In [22]:
df=pd.DataFrame({'Age':[20,25]})
print(df.rename(columns={'Age':'Student_Age'}))

   Student_Age
0           20
1           25


---

## 14. `astype` — changing a column's type

```python
df['Age'] = df['Age'].astype(int)
```

The output is the whole lesson:

```text
   before   Age    str        <- '20' and '25' are TEXT
   after    Age    int64
```

They looked like numbers when printed. They were not. Text sorts as `'100' < '20'`, and
`df['Age'].mean()` on text fails or concatenates. This is why `info()` in section 6 comes
before any analysis: **check the dtype, do not trust the printed value.**

Numbers arriving as text is the normal case, not the exception. CSV has no types, so
`'20 '` with a trailing space, `'1,200'` with a comma, or one `'unknown'` in ten thousand
rows will hold the whole column at `str`.

### The assignment is doing real work

```python
df['Age'] = df['Age'].astype(int)
```

`astype` returns a **new Series**. Without `df['Age'] =`, nothing changes. Writing to
`df['Age']` replaces the column in place — and the same syntax creates a column when the
name is new, which is exactly how section 17 adds `Rank`.

### When `astype` refuses

```python
pd.Series(['20','abc']).astype(int)     # ValueError
pd.to_numeric(df['Age'], errors='coerce')   # 'abc' becomes NaN instead of crashing
```

`to_numeric` with `errors='coerce'` is the practical tool on messy files: bad values
become `NaN`, and section 11 handles them from there.

**Takeaway:** `astype` converts a column and returns a new one — assign it back. Check
dtypes early, because text that looks numeric breaks everything downstream.


In [23]:
df=pd.DataFrame({'Age':['20','25']})
print(df.dtypes)
df['Age']=df['Age'].astype(int)
print(df.dtypes)

Age    str
dtype: object
Age    int64
dtype: object


---

## 15. `replace` — swapping values

```python
df.replace('Pune','Nashik')       # both Pune rows become Nashik
```

`rename` changed a **column name**; `replace` changes **values inside the data**. Easy to
mix up, completely different jobs.

Applied to the whole frame it searches every column. Aim it at one column when you can:

```python
df['City'] = df['City'].replace('Pune','Nashik')       # safer
df['City'] = df['City'].replace({'Pune':'Nashik', 'Mumbai':'Thane'})   # many at once
```

Both `Pune` rows changed from one call — `replace` is not "replace the first match", it
is every match.

This is the everyday tool for cleaning inconsistent categories: `'pune'`, `'PUNE'`, and
`'Pune '` are three different values to pandas, and they will produce three separate
groups in section 18 unless you normalise them first.

**Returns a copy.** Same rule as the last two sections.

**Takeaway:** `replace` edits values, `rename` edits labels. Normalise your categories
before you group by them.


In [24]:
df=pd.DataFrame({'City':['Pune','Mumbai','Pune']})
print(df.replace('Pune','Nashik'))

     City
0  Nashik
1  Mumbai
2  Nashik


---

## 16. `get_dummies` — text into numbers

```python
pd.get_dummies(df['City'])
```

```text
   City          Delhi  Mumbai   Pune
   Pune          False   False   True         one column per distinct value,
   Mumbai        False    True  False         exactly one True per row
   Delhi          True   False  False
```

This is **one-hot encoding**, and it exists because models do maths. A model cannot
multiply `'Pune'` by a weight. Three text values become three true/false columns it can.

### Why not just number them?

`Pune=1, Mumbai=2, Delhi=3` would compile and be wrong. It tells the model
`Delhi > Mumbai > Pune` and that `Mumbai` is the average of the other two. None of that is
true — city names have no order. One-hot avoids inventing an order that is not there.

For values that **do** have an order — `small < medium < large` — numbering is correct,
and that is called ordinal encoding.

### Two details

**The output is `bool`,** not `1` and `0`. Verified. Modern pandas returns booleans;
older tutorials show integers. Models accept both, since `True` is `1`. Force it with
`dtype=int` if you prefer seeing numbers.

**`drop_first=True`** is the argument you will meet in the ML phase:

```python
pd.get_dummies(df['City'], drop_first=True)     # Delhi column dropped
```

With `Mumbai=False` and `Pune=False`, the row must be Delhi — the third column carries no
new information. Keeping all three makes the columns perfectly correlated, which is the
singular-matrix problem from the NumPy notebook's `np.linalg` section, arriving in real
data. Linear models care; trees do not.

**Takeaway:** `get_dummies` turns categories into true/false columns so a model can use
them, without inventing an order.


In [25]:
df=pd.DataFrame({'City':['Pune','Mumbai','Delhi']})
print(pd.get_dummies(df['City']))

   Delhi  Mumbai   Pune
0  False   False   True
1  False    True  False
2   True   False  False


---

## 17. `sort_values` and `rank`

```python
df.sort_values('Marks')          # 50, 70, 80
df['Rank'] = df['Marks'].rank(ascending=False)
```

### Read the two outputs together

```text
   sort_values('Marks')          then the frame with Rank
      Marks                        Marks  Rank
   0     50                     0     50   3.0
   2     70                     1     80   1.0
   1     80                     2     70   2.0
```

The first print is sorted. The second is back in the **original order** — because
`sort_values` returned a copy that was printed and thrown away. `df` was never sorted.
Reassign if you want it kept.

Notice the sorted output's labels: `0, 2, 1`. Labels travel with their rows. Section 8,
one more time.

### What `rank` gives you

`rank` does not reorder anything. It adds **the position each row would have**, in place.
That is why it works as a new column: 80 is 1st, 70 is 2nd, 50 is 3rd, each number
staying on its own row.

`ascending=False` makes the highest mark rank 1 — which is what "rank" means for scores.
The default is ascending, where the smallest value gets rank 1.

### Ties are averaged

The results are floats — `3.0`, not `3` — because of ties. Verified with a tie added:

```python
pd.Series([50,80,80,70]).rank(ascending=False)   # -> [4.0, 1.5, 1.5, 3.0]
```

Two rows share 1st and 2nd, so both get `1.5`. That is `method='average'`, the default.
`method='min'` gives both `1.0` (the sports convention) and `method='dense'` avoids the
gap. Choose deliberately when ties matter.

**Sorting by several columns:**

```python
df.sort_values(['City','Marks'], ascending=[True, False])
```

**Takeaway:** `sort_values` reorders and returns a copy; `rank` labels each row in place.
Ties get the average rank unless you say otherwise.


In [26]:
df=pd.DataFrame({'Marks':[50,80,70]})
print(df.sort_values('Marks'))
df['Rank']=df['Marks'].rank(ascending=False)
print(df)

   Marks
0     50
2     70
1     80
   Marks  Rank
0     50   3.0
1     80   1.0
2     70   2.0


---

## 18. `groupby` — split, apply, combine

```python
df.groupby('City')['Sales'].sum()
```

Three steps, always:

```text
   split                     apply             combine
   -----                     -----             -------
   Pune   -> [100, 200]      sum -> 300        City
   Mumbai -> [300]           sum -> 300        Mumbai    300
                                               Pune      300
```

Read the call left to right: **group by `City`, take the `Sales` column, sum it.**

### `sum` and `mean` tell different stories here

```text
   sum                       mean
   Mumbai    300             Mumbai    300.0
   Pune      300             Pune      150.0
```

Identical totals, very different businesses. Pune made 300 across **two** sales; Mumbai
made 300 in **one**. Always check `count()` alongside a total — a large sum built from one
row is a different fact from the same sum built from fifty.

```python
df.groupby('City')['Sales'].agg(['sum','mean','count'])     # all three at once
```

### Two details in the output

**The result is a Series, and `City` became its index.** The group key always becomes the
index. `.reset_index()` turns it back into a normal column when you need a plain frame.

**Mumbai printed first.** The input had Pune first — `groupby` **sorts the keys** by
default. Verified: `sort=False` preserves first-appearance order, and is faster on large
frames.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> ```sql
> SELECT City, SUM(Sales) FROM df GROUP BY City;
> ```
>
> Line for line, that is this cell. The mental model transfers completely: `groupby` is
> `GROUP BY`, `.agg` is the aggregate list, and filtering before the groupby is `WHERE`
> while filtering after it is `HAVING`.

**Takeaway:** `groupby` splits rows by a key, runs a function on each group, and combines
the answers. The key becomes the index, and you should almost always look at `count` next
to your total.


In [27]:
df=pd.DataFrame({'City':['Pune','Pune','Mumbai'],'Sales':[100,200,300]})
print(df.groupby('City')['Sales'].sum())
print(df.groupby('City')['Sales'].mean())

City
Mumbai    300
Pune      300
Name: Sales, dtype: int64
City
Mumbai    300.0
Pune      150.0
Name: Sales, dtype: float64


---

## 19. `pivot_table` — groupby laid out as a grid

```python
pd.pivot_table(df, values='Sales', index='City', aggfunc='sum')
```

The numbers are identical to section 18 — same split, same sum. Two differences:

| | `groupby` | `pivot_table` |
|---|---|---|
| Returns here | a **Series** | a **DataFrame** (note `Sales` as a column header) |
| Built for | one key, any function | **two** keys — one down, one across |

The reason `pivot_table` exists is the second row of that table:

```python
pd.pivot_table(df, values='Sales', index='City', columns='Month', aggfunc='sum')
```

```text
              Jan    Feb    Mar          index=  rows
   Mumbai     300    250    400          columns= the grid across the top
   Pune       150    180    210          values=  what fills the cells
```

That is a cross-tab — the Excel pivot table, same idea and nearly the same words. Doing it
with `groupby` needs a `.unstack()` afterwards.

Two useful arguments: `aggfunc` accepts a list (`['sum','mean']`), and `fill_value=0`
replaces the `NaN` that appears wherever a combination has no rows.

**Takeaway:** one grouping key — `groupby`. Two keys, laid out as a grid — `pivot_table`.


In [28]:
df=pd.DataFrame({'City':['Pune','Pune','Mumbai'],'Sales':[100,200,300]})
print(pd.pivot_table(df,values='Sales',index='City',aggfunc='sum'))

        Sales
City         
Mumbai    300
Pune      300


---

## 20. Dates — `to_datetime` and the `.dt` accessor

```python
dates['Date'] = pd.to_datetime(dates['Date'])
dates['Date'].dt.year        # 2025, 2025
dates['Date'].dt.month       # 1, 2
```

The frame starts with `'2025-01-01'` — **text that looks like a date**. Pandas cannot sort
it as time, subtract it, or resample it until you convert. `to_datetime` parses the string
into a real `datetime64` value.

### What `.dt` is

Once the column is a datetime, `.dt` unlocks its parts:

```python
.dt.year   .dt.month   .dt.day   .dt.hour
.dt.dayofweek          # Monday = 0
.dt.day_name()         # 'Wednesday'
.dt.quarter
```

`.dt` only exists on datetime columns. Calling it on text raises
`AttributeError: Can only use .dt accessor with datetimelike values` — which is the
reminder that you skipped `to_datetime`.

Note the output dtype: `int32`. `.dt.year` returns numbers you can group by, filter on,
and plot — which is the point. `df.groupby(df['Date'].dt.month)['Sales'].sum()` is
monthly sales in one line.

### One argument worth knowing now

```python
pd.to_datetime(col, format='%d/%m/%Y')     # be explicit
```

`01/02/2025` is 1 February in India and 2 January in the US. Pandas guesses when you do
not say, and a wrong guess corrupts the data silently — no error, just wrong months. Pass
`format` whenever the source is not ISO `YYYY-MM-DD`.

**Takeaway:** convert once with `to_datetime`, then use `.dt` to pull out the parts you
want to group or filter by.


In [29]:
dates=pd.DataFrame({'Date':['2025-01-01','2025-02-01']})
dates['Date']=pd.to_datetime(dates['Date'])
print(dates['Date'].dt.year)
print(dates['Date'].dt.month)

0    2025
1    2025
Name: Date, dtype: int32
0    1
1    2
Name: Date, dtype: int32


---

## 21. Text — the `.str` accessor

```python
s.str.upper()
s.str.contains('a', case=False)
s.str.len()
```

Same idea as `.dt`, for text. `.str` applies a Python string operation to **every value in
the column**, with no loop.

```text
   s.str.upper()                ALICE, BOB, CHARLIE
   s.str.contains('a', case=False)   True, False, True
   s.str.len()                  5, 3, 7
```

### Why `case=False` matters here

Without it, `contains('a')` would miss `'Alice'` — the `A` is capital. Real text data is
full of this: `'Pune'`, `'pune'`, and `'PUNE'` are three separate values to pandas, and
they become three separate groups in a `groupby`. The standard first move on any text
column:

```python
df['City'] = df['City'].str.strip().str.lower()
```

`strip()` removes stray spaces, `lower()` removes case differences. Chaining `.str` twice
is normal — each call returns a Series, so the next `.str` applies to that.

### The `.str` calls you will use most

| Call | Does |
|---|---|
| `.str.strip()` | trim whitespace — fixes invisible bugs |
| `.str.lower()` / `.str.upper()` | normalise case |
| `.str.contains(p)` | True/False mask — pairs with section 9 |
| `.str.replace(a, b)` | substring replacement |
| `.str.split(',')` | split into lists |
| `.str.startswith(p)` | prefix test |

`.str.contains` is the one that connects back: it returns a mask, so
`df[df['City'].str.contains('pun', case=False)]` filters rows by text.

### The accessor pattern

You have now seen three: `.str` for text, `.dt` for dates, and `.cat` for categories.
Pandas groups type-specific operations behind an accessor so `df.something` does not have
a thousand methods on it. When you want a text operation, the answer starts with `.str`.

**Missing values note:** `.str` returns `NaN` for missing entries rather than raising —
handy, but it means a `contains` mask can hold `NaN` instead of `False`. Add
`na=False` when you feed it straight into a filter.

**Takeaway:** `.str` runs string operations over a whole column. Strip and lower your text
columns before grouping, or you will count the same category twice.


In [30]:
s=pd.Series(['Alice','Bob','Charlie'])
print(s.str.upper())
print(s.str.lower())
print(s.str.contains('a',case=False))
print(s.str.len())

0      ALICE
1        BOB
2    CHARLIE
dtype: str
0      alice
1        bob
2    charlie
dtype: str
0     True
1    False
2     True
dtype: bool
0    5
1    3
2    7
dtype: int64


---

## Recap — what this notebook added

| Section | You can now |
|---|---|
| 2-3 | Explain why `df.shape` has no parentheses and `df.head()` does |
| 4-5 | Read and write CSV/Excel/JSON, and write CSV without a junk index column |
| 6 | Open an unknown dataset with `info()`, `shape`, and `describe()` |
| 7-8 | Choose `loc` or `iloc` deliberately, and explain a `KeyError` |
| 9-10 | Filter rows with a mask or with `query` |
| 11-12 | Find and handle missing values and duplicates |
| 13-17 | Rename, convert types, replace values, one-hot encode, sort, and rank |
| 18-19 | Aggregate with `groupby`, and cross-tabulate with `pivot_table` |
| 20-21 | Work with dates through `.dt` and text through `.str` |

### The five gotchas worth memorising

1. **No parentheses = a fact** (`df.shape`). **Parentheses = an action** (`df.head()`).
   `<bound method ...>` in your output means you forgot the `()`.
2. **Almost every method returns a copy.** `fillna`, `rename`, `sort_values`,
   `drop_duplicates`, `replace` — none of them change `df`. Assign the result back.
3. **`loc` is labels, `iloc` is positions**, and after a filter or sort they no longer
   match. `KeyError: 1` means "no label 1", not "no row 1".
4. **One missing value turns an int column into floats**, because `NaN` is a float.
5. **`to_csv` writes the index by default** — pass `index=False` unless the index is real
   data.

### Two cells to fix when you revisit

- The read cell runs **before** the write cell that creates the file, so it raises
  `FileNotFoundError`. Move it below, or create the file first.
- `to_excel` needs `openpyxl` (`pip install openpyxl`, then restart the kernel). Until
  then `to_json` on the line after it never runs.

Both errors are left in place on purpose — they are worth more explained than deleted.

### Not covered yet

`merge` and `concat` (they are in [`example_01.ipynb`](./example_01.ipynb)), `apply` and
`map`, `value_counts`, `nunique`, multi-column `groupby`, `agg` with several functions,
`MultiIndex`, `resample` for time series, `crosstab`, `cut` for binning, and
`pd.options.display` settings. The [`DataSet/`](./DataSet/) folder holds real CSVs to try
these on.

**If you remember only one thing:** a DataFrame is **labelled columns**, and nearly every
method hands you a **copy**. Get those two into your fingers — assign the result back, and
never assume the index is still `0, 1, 2` — and most Pandas confusion disappears.
